In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [2]:
df = pd.read_csv(r'C:\Users\Compumisr\Desktop\ANN_calssification\Churn_Modelling.csv')
df.head(5)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
df = df.drop(['RowNumber','CustomerId','Surname'],axis=1)
df.head(5)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])

In [5]:
df.head(5)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [6]:
df['Geography'].value_counts()

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

In [7]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder()
geo_encoder = ohe.fit_transform(df[['Geography']]).toarray()
ohe.get_feature_names_out(['Geography'])
geo_df = pd.DataFrame(geo_encoder,columns=ohe.get_feature_names_out(['Geography']))

In [8]:
df = pd.concat([df.drop('Geography',axis=1),geo_df],axis=1)
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [9]:
with open('label_encoder_gender.pkl','wb') as file:    
    pickle.dump(le, file)
    
    with open('one_hot_encoder_geography.pkl','wb') as file:    
        pickle.dump(ohe, file)   

In [10]:
X = df.drop('Exited',axis=1)
y = df['Exited']  
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)  

In [11]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler, file)

In [12]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [13]:
(X_train.shape[1],)

(12,)

In [14]:
model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)), 
    Dense(32,activation='relu'), 
    Dense(1,activation='sigmoid')  
]

)

In [15]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [16]:
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss=tensorflow.keras.losses.BinaryCrossentropy()
loss

In [17]:

model.compile(optimizer=opt,loss="binary_crossentropy",metrics=['accuracy'])

In [18]:
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard

log_dir="logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [19]:
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)


In [20]:
history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

Epoch 1/100


250/250 [==============================] - 5s 9ms/step - loss: 0.3977 - accuracy: 0.8320 - val_loss: 0.3544 - val_accuracy: 0.8565
Epoch 2/100
250/250 [==============================] - 1s 6ms/step - loss: 0.3522 - accuracy: 0.8562 - val_loss: 0.3561 - val_accuracy: 0.8575
Epoch 3/100
250/250 [==============================] - 1s 5ms/step - loss: 0.3511 - accuracy: 0.8569 - val_loss: 0.3427 - val_accuracy: 0.8600
Epoch 4/100
250/250 [==============================] - 1s 6ms/step - loss: 0.3468 - accuracy: 0.8577 - val_loss: 0.3544 - val_accuracy: 0.8565
Epoch 5/100
250/250 [==============================] - 1s 5ms/step - loss: 0.3408 - accuracy: 0.8601 - val_loss: 0.3624 - val_accuracy: 0.8530
Epoch 6/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3403 - accuracy: 0.8639 - val_loss: 0.3463 - val_accuracy: 0.8590
Epoch 7/100
250/250 [==============================] - 1s 5ms/step - loss: 0.3356 - accuracy: 0.8643 - val_loss: 0.3628 - val_accuracy: 0.85

In [21]:
model.save('model.h5')

c:\Users\Compumisr\Desktop\ANN_calssification\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
